### 1. Exploitation Zone: Multi-Modal Semantic Vectorization via OpenAI CLIP
In this exploitation stage, we upgrade our image feature extraction standard from traditional CNNs to **OpenAI's CLIP (Contrastive Language-Image Pre-training)**. 

CLIP maps both visual assets and natural language queries into a unified, shared high-dimensional embedding space. This architectural choice empowers our **Milvus Vector Engine** to support not only Content-Based Image Retrieval (CBIR / Image-to-Image) but also **Natural Language Cross-Modal Search (Text-to-Image)** out-of-the-box.

#### Pipeline Blueprint:
1. **Distributed Transformer Inference:** PySpark orchestration leverages `mapPartitions()` to load the `transformers.CLIPModel` locally on worker nodes, processing localized image binary streams with minimum network payload overhead.
2. **ACID-Backed Gold Repository:** Extracted 512-dimensional floating-point vectors are simultaneously streamed into **Milvus Standalone** (for low-latency online vector searches) and committed as a **Delta Lake Gold Table** (for disaster recovery, offline compliance, and ML training runs).

In [1]:
import os
import io
import boto3
from pymilvus import MilvusClient, FieldSchema, CollectionSchema, DataType
from pyspark.sql import SparkSession

# Initialize MinIO coordinates
endpoint = os.getenv("MINIO_ENDPOINT")
access_key = os.getenv("MINIO_ACCESS_KEY")
secret_key = os.getenv("MINIO_SECRET_KEY")

# Initialize SparkSession with Delta Lake and S3A support
DELTA_VERSION = "4.1.0"
spark = SparkSession.builder \
    .appName("exploitation_zone_vectorization") \
    .master("spark://spark-master:7077") \
    .config("spark.jars.packages", f"org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,io.delta:delta-spark_2.13:{DELTA_VERSION}") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", endpoint) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.access.key", access_key) \
    .config("spark.hadoop.fs.s3a.secret.key", secret_key) \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

# Normalize Hadoop configuration values (e.g., converting "60s" to "60") to prevent version mismatch errors
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
for item in hadoop_conf.iterator():
    key = item.getKey()
    value = item.getValue()

    if isinstance(value, str) and (value.endswith("s") or value.endswith("h")):
        numeric_value = "".join(char for char in value if char.isdigit())
        hadoop_conf.set(key, numeric_value)
# Broadcast credentials for worker nodes to avoid serialization issues
minio_creds = {"endpoint": endpoint, "access_key": access_key, "secret_key": secret_key}
credentials_broadcast = spark.sparkContext.broadcast(minio_creds)


### 2. Vector Database (Milvus) Schema Definition
Before data ingestion, we initialize the Milvus collection and define a schema tailored for CLIP's 512-dimensional vector output. We include an idempotency check (dropping the collection if it already exists) to ensure reproducible pipeline runs without data duplication.

In [2]:
MILVUS_URI = "http://milvus:19530"
COLLECTION_NAME = "image_vector_catalog"
EMBEDDING_DIM = 512

milvus_client = MilvusClient(MILVUS_URI)

# Idempotency check: recreate collection if it exists
if milvus_client.has_collection(collection_name=COLLECTION_NAME):
    milvus_client.drop_collection(collection_name=COLLECTION_NAME)

fields = [
    FieldSchema(name="id", dtype=DataType.VARCHAR, max_length=64, is_primary=True, description="Unique Image ID"),
    FieldSchema(name="embeddings", dtype=DataType.FLOAT_VECTOR, dim=EMBEDDING_DIM, description="CLIP Latent Embeddings"),
    FieldSchema(name="label", dtype=DataType.VARCHAR, max_length=128, description="Domain Label")
]

schema = CollectionSchema(fields, description="Multi-Modal Visual Catalog")
milvus_client.create_collection(collection_name=COLLECTION_NAME, schema=schema)

print(f"Milvus Vector Schema Materialized: {COLLECTION_NAME} (Dim: {EMBEDDING_DIM})")

Milvus Vector Schema Materialized: image_vector_catalog (Dim: 512)


### 3. Distributed Inference & Dual-Write Pipeline
After caching the CLIP model locally, we construct and trigger the Spark `mapPartitions` execution graph. Data flows into Delta Lake to form a trusted feature store while simultaneously being dispatched in batches to Milvus for indexing.

In [3]:
from transformers import CLIPProcessor, CLIPModel
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, FloatType

# Download Model to a shared/local path for workers
local_model_path = "/opt/spark/hf_cache/clip_local_model"
print(f"[INFO] Downloading CLIP model to {local_model_path}...")
CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32").save_pretrained(local_model_path)
CLIPModel.from_pretrained("openai/clip-vit-base-patch32").save_pretrained(local_model_path)

def generate_clip_embeddings_and_catalog(rows):
    import io, boto3, torch
    from PIL import Image
    from transformers import CLIPProcessor, CLIPModel
    from pymilvus import MilvusClient
    
    milvus_client = MilvusClient("http://milvus:19530")
    creds = credentials_broadcast.value
    s3_client = boto3.client("s3", endpoint_url=creds["endpoint"], aws_access_key_id=creds["access_key"], aws_secret_access_key=creds["secret_key"])
    
    processor = CLIPProcessor.from_pretrained(local_model_path)
    model = CLIPModel.from_pretrained(local_model_path)
    milvus_batch = []
    
    for row in rows:
        try:
            # Fetch image from S3 using broadcasted credentials
            bucket, key = row["trusted_path"].replace("s3a://", "").split("/", 1)
            obj = s3_client.get_object(Bucket=bucket, Key=key)
            img = Image.open(io.BytesIO(obj["Body"].read()))
            
            # Generate normalized CLIP embeddings
            inputs = processor(images=img, return_tensors="pt")
            with torch.no_grad():
                vision_outputs = model.vision_model(**inputs)
                image_features = model.visual_projection(vision_outputs.pooler_output)
                image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True) # L2 Normalization
                embedding = image_features.squeeze(0).tolist() 
            
            # Prepare batch payload for Milvus
            milvus_batch.append({"id": str(row["id"]), "embeddings": embedding, "label": str(row["label"])})
            
            # Yield data for downstream Delta Lake ingestion
            yield (str(row["id"]), embedding, str(row["label"]), row["trusted_path"])
            
            # Flush batch to Milvus to prevent OOM errors
            if len(milvus_batch) >= 100:
                milvus_client.insert(collection_name=COLLECTION_NAME, data=milvus_batch)
                milvus_batch = []
        except Exception:
            # Fault tolerance: skip corrupt images without failing the entire partition
            continue
            
    # Flush any remaining records
    if milvus_batch:
        milvus_client.insert(collection_name=COLLECTION_NAME, data=milvus_batch)

# Define DAG schema
exploitation_schema = StructType([
    StructField("id", StringType(), True),                         
    StructField("embeddings", ArrayType(FloatType()), True),       
    StructField("label", StringType(), True),                      
    StructField("trusted_s3_path", StringType(), True)             
])

# Read source data and apply distributed transformation
trusted_df = spark.read.format("delta").load("s3a://trusted-zone/file_catalog/")
exploitation_rdd = trusted_df.repartition(8).rdd.mapPartitions(generate_clip_embeddings_and_catalog)
exploitation_catalog_df = spark.createDataFrame(exploitation_rdd, schema=exploitation_schema)

# Execute Action: Write to Delta Lake
exploitation_delta_path = "s3a://exploitation-zone/image_catalog"
print("Submitting distributed graph and writing to Delta Lake/Milvus...")
exploitation_catalog_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").save(exploitation_delta_path)
print("Fetching ingestion metrics from Delta storage...")
actual_ingested_count = spark.read.format("delta").load(exploitation_delta_path).count()
print(f"Ingestion successful! Normalized {actual_ingested_count} records.")
# Build Milvus Index post-ingestion for optimal search performance
index_params = milvus_client.prepare_index_params()
index_params.add_index(field_name="embeddings", metric_type="L2", index_type="IVF_FLAT", params={"nlist": 128})
milvus_client.create_index(collection_name=COLLECTION_NAME, index_params=index_params)
milvus_client.load_collection(collection_name=COLLECTION_NAME)

print("Pipeline execution complete. Milvus Index built and loaded.")

[INFO] Downloading CLIP model to /opt/spark/hf_cache/clip_local_model...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Submitting distributed graph and writing to Delta Lake/Milvus...
Fetching ingestion metrics from Delta storage...
Ingestion successful! Normalized 6674 records.
Pipeline execution complete. Milvus Index built and loaded.


### 4. Retrieval Validation Phase
To verify end-to-end functionality, we sample a random record from our Delta Lake feature store as a query vector and test the Milvus vector search. The top result should be an exact self-match (Distance approaching 0.0), proving data integrity across both systems.

In [4]:
import pyspark.sql.functions as F
gold_df = spark.read.format("delta").load("s3a://exploitation-zone/image_catalog")
# Fetch a random test embedding from the offline Feature Store
test_sample = gold_df.sample(withReplacement=False, fraction=0.05).limit(1).collect()[0]
test_id, test_label, test_vector = test_sample["id"], test_sample["label"], test_sample["embeddings"]

print(f"Testing semantic search for ID: {test_id} (Label: {test_label})")

# Execute Vector Search
results = milvus_client.search(
    collection_name=COLLECTION_NAME,
    data=[test_vector],                  
    anns_field="embeddings",             
    search_params={"metric_type": "L2", "params": {"nprobe": 10}},
    limit=3,                             
    output_fields=["label", "id"]        
)

# Parse and validate results
for hits in results:
    for i, hit in enumerate(hits):
        returned_id = hit["entity"].get("id")
        returned_label = hit["entity"].get("label")
        
        match_status = "EXACT MATCH" if str(returned_id) == str(test_id) else "NEIGHBOR"
        print(f"Rank {i+1} | {match_status} | ID: {returned_id} | Label: {returned_label} | Distance: {hit['distance']:.6f}")

print("Exploitation Pipeline Validation Completed.")

Testing semantic search for ID: image_1778924731342.jpg (Label: rain)
Rank 1 | EXACT MATCH | ID: image_1778924731342.jpg | Label: rain | Distance: 0.000000
Rank 2 | NEIGHBOR | ID: image_1778924731668.jpg | Label: rain | Distance: 0.220960
Rank 3 | NEIGHBOR | ID: image_1778924755911.jpg | Label: rain | Distance: 0.225519
Exploitation Pipeline Validation Completed.


In [ ]:
spark.stop()